# Denver, Colorado — Land Value Tax Policy Analysis

**City & County of Denver** is a consolidated city-and-county (Census FIPS **08031**).
A Denver property-tax bill is a *stack* of levies from many independent taxing
entities — Denver Public Schools, the City & County of Denver itself, Urban
Drainage & Flood Control (Mile High Flood District), the Regional Transportation
District, Denver Health, metro districts, and BIDs/GIDs/the DDA. This model
touches **only the City & County of Denver's own municipal mill levy** and holds
every other entity's levy unchanged.

## Two scenarios modeled (custom-scoped Landslide run)

| Parameter | Scenario A | Scenario B |
|---|---|---|
| Reform | Split-rate (two-rate) property tax | Split-rate (two-rate) property tax |
| Land : improvement ratio | **4 : 1** | **4 : 1** |
| Geographic scope | **Citywide** | **Downtown Denver Business Improvement District only** |
| Levy scope | City & County of Denver's own mills | City & County of Denver's own mills |
| Revenue neutrality | Citywide, to Denver's own municipal levy | **Within the BID** — BID parcels' aggregate municipal contribution held constant; every parcel outside the BID keeps its current municipal tax exactly |
| Exemptions | All existing exemptions preserved | All existing exemptions preserved |

Fully-exempt parcels are held out of the solver and excluded from the export and
report charts (they carry no signal); they are reported by count.

## Colorado's assessment machinery

Colorado taxes an *assessed* value derived from the assessor's *actual* (market)
value, and since the post-Gallagher reforms the assessment rate **differs between
school-district levies and other local-government levies**. Denver's own
municipal levy is a **non-school local-government levy**, so the *local* rate and
the *local* assessed values are the correct base for this model.

Denver's published parcel file already carries the assessor's own
`ASSESSED_LAND_VALUE_LOCAL` / `ASSESSED_BLDG_VALUE_LOCAL` / `EXEMPT_AMT_LOCAL`
columns, so **this model does not have to assume a statutory assessment rate** —
it uses the assessor's arithmetic directly. Section 3 nevertheless *reverse-engineers*
the statutory formula from those columns and reports the exact fit, both as a
data-integrity check and because the formula is needed to fill in the parcels
where the GIS layer left the assessed columns null (see Section 4).

## Data sources

| What | Source |
|---|---|
| Parcels + assessor values + exemptions + tax district | City & County of Denver Open Data, `ODC_PROP_PARCELS_A` FeatureServer **layer 245** — `https://services1.arcgis.com/zdB7qR0BtYrg0Xpl/arcgis/rest/services/ODC_PROP_PARCELS_A/FeatureServer/245` (240,366 features) |
| Downtown Denver BID boundary | Denver Open Data, `ODC_ADMN_BUSINESSIMPROVDIST_A` layer 1, `DISTRICT_NAME = 'DOWNTOWN DENVER BUSINESS IMPROVEMENT DISTRICT'` |
| Downtown Development Authority (comparison only) | Denver Open Data, `Downtown_Development_Authority` layer 0 |
| Statistical neighborhoods (comparison only) | Denver Open Data, `ODC_ADMN_NEIGHBORHOOD_A` layer 13 |
| Demographics | Census ACS 2022 block groups, county FIPS 08031 |

## Section 1 — Imports, constants, `.env`

In [ ]:
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, str(Path("../..").resolve()))
REPO_ROOT = Path("../..").resolve()
load_dotenv(REPO_ROOT / ".env")

from lvt.lvt_utils import (
    calculate_category_tax_summary,
    calculate_current_tax,
    model_split_rate_tax,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.policy_analysis import analyze_land_by_improvement_share

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

CITY_NAME = "denver"
STATE_FIPS = "08"
COUNTY_FIPS = "031"          # Denver County == City & County of Denver
LAND_IMPROVEMENT_RATIO = 4.0
MODEL_TYPE = "split_rate:4.0"

# ---- Roll years ----
# The parcel file is the CURRENT roll (2026, payable 2027) — Section 3 proves this
# by fitting the statutory formula exactly. The most recent CERTIFIED levy is roll
# 2025 (payable 2026); no roll-2026 certification exists yet. See Section 6.
ROLL_YEAR_PARCELS = 2026
ROLL_YEAR_LEVY = 2025

# ---- Official figures: City & County of Denver's OWN levy, roll 2025 ----
# Source: Denver Assessor, "2025 Abstract of Assessment and Summary of Levies"
#   https://www.denvergov.org/files/assets/public/v/1/finance/documents/assessor/2026/2025-abstract.pdf
DENVER_OWN_MILLS_PUBLISHED = 26.328          # Denver's own mills, roll 2025
OFFICIAL_CITY_LEVY = 669_926_384.0           # Denver's own levy, net of TIF diversion
DENVER_GROSS_AV_2025_LOCAL = 26_045_588_170.0   # gross assessed value, local-govt basis
DENVER_TOTAL_ACTUAL_VALUE_2025 = 244_647_640_439.0  # total actual value, local-govt basis
TOTAL_STACKED_MILLS_2025 = 79.602            # minimum total levy (Denver + DPS + drainage)

# Denver's own sub-levies, roll 2025 (sum = 26.328): General Fund 9.628,
# Bond Principal 4.768, Bond Interest 1.732, Social Services 2.473,
# Developmentally Disabled 1.030, Fire Pension 0.994, Police Pension 1.185,
# Capital Maintenance 2.576, Affordable Housing 0.398, Library 1.544.
# There is NO Denver Health, Denver Preschool Program, or RTD property-tax mill.

# ---- Parcel-map identity columns ----
PARCEL_ID_COL = "SCHEDNUM"
OWNER_NAME_COL = "OWNER_NAME"
OWNER_ADDRESS_COL = "OWNER_ADDRESS_LINE1"
PARCEL_URL_TEMPLATE = "https://www.denvergov.org/property/realproperty/summary/{parcel_id}"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARCEL_PATH = DATA_DIR / "parcels_raw.gpq"

ARCGIS = "https://services1.arcgis.com/zdB7qR0BtYrg0Xpl/arcgis/rest/services"
PARCEL_LAYER = f"{ARCGIS}/ODC_PROP_PARCELS_A/FeatureServer/245"

print(f"City: {CITY_NAME}   FIPS: {STATE_FIPS + COUNTY_FIPS}   Model: {MODEL_TYPE}")
print(f"CENSUS_API_KEY present: {bool(os.getenv('CENSUS_API_KEY'))}")

## Section 2 — Load parcel data

`ODC_PROP_PARCELS_A` layer 245 is Denver's assessor parcel layer: one row per
schedule number, with appraised (actual) land/improvement values, the assessor's
own assessed and exempt amounts split **local vs. school**, the tax-district
code, Denver's own detailed class code (`D_CLASS_CN` / `DCL12`), land area,
building area, and unit counts.

Fetched as **geoJSON** so holes and multi-part parcels survive — the repo's
generic ArcGIS helper keeps only `rings[0]`. Cached to `data/parcels_raw.gpq`.

In [ ]:
PARCEL_FIELDS = [
    "OBJECTID", "SCHEDNUM", "MAPNUM", "PARCEL_SOURCE", "SYSTEM_START_DATE",
    "OWNER_NAME", "OWNER_ADDRESS_LINE1", "OWNER_CITY", "OWNER_STATE",
    "SITUS_ADDRESS_LINE1", "SITUS_CITY", "SITUS_ZIP",
    "TAX_DIST", "PROP_CLASS", "D_CLASS", "D_CLASS_CN", "DCL12",
    "ZONE_ID", "ZONE_10",
    "APPRAISED_LAND_VALUE", "APPRAISED_IMP_VALUE", "APPRAISED_TOTAL_VALUE",
    "ASSESSED_LAND_VALUE_LOCAL", "ASSESSED_BLDG_VALUE_LOCAL",
    "ASSESSED_TOTAL_VALUE_LOCAL", "EXEMPT_AMT_LOCAL", "TAXABLE_AMT_LOCAL",
    "ASSESSED_LAND_VALUE_SCH", "ASSESSED_BLDG_VALUE_SCH",
    "ASSESSED_TOTAL_VALUE_SCH", "EXEMPT_AMT_SCH", "TAXABLE_AMT_SCH",
    "LAND_AREA", "RES_ORIG_YEAR_BUILT", "RES_ABOVE_GRADE_AREA",
    "COM_ORIG_YEAR_BUILT", "COM_GROSS_AREA", "COM_NET_AREA",
    "COM_STRUCTURE_TYPE", "TOT_UNITS", "SALE_YEAR", "SALE_PRICE",
]


def _fetch_page(url, params, attempt=0):
    try:
        r = requests.get(url, params=params, timeout=300)
        r.raise_for_status()
        js = r.json()
        if "features" not in js:
            raise RuntimeError(f"no features: {str(js)[:200]}")
        return js["features"]
    except Exception:
        if attempt >= 5:
            raise
        time.sleep(3 * (attempt + 1))
        return _fetch_page(url, params, attempt + 1)


def fetch_arcgis_geojson(layer_url, fields="*", where="1=1", page=2000, workers=8,
                         order_by="OBJECTID"):
    """Paginated parallel geoJSON download of an ArcGIS FeatureServer layer."""
    q = f"{layer_url}/query"
    n = requests.get(
        q, params={"where": where, "returnCountOnly": "true", "f": "json"}, timeout=120
    ).json()["count"]
    offsets = list(range(0, n, page))
    print(f"  {n:,} features -> {len(offsets)} pages")

    def one(off):
        return off, _fetch_page(q, {
            "where": where,
            "outFields": fields if isinstance(fields, str) else ",".join(fields),
            "returnGeometry": "true", "orderByFields": order_by,
            "resultOffset": off, "resultRecordCount": page,
            "outSR": 4326, "geometryPrecision": 7, "f": "geojson",
        })

    pages = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(one, o) for o in offsets]):
            off, feats = fut.result()
            pages[off] = feats
    feats = [f for o in offsets for f in pages[o]]
    gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
    gdf["geometry"] = gdf.geometry.make_valid()
    return gdf


if PARCEL_PATH.exists():
    gdf_raw = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf_raw):,} parcels from cache ({PARCEL_PATH})")
else:
    print("Downloading Denver assessor parcels...")
    gdf_raw = fetch_arcgis_geojson(PARCEL_LAYER, PARCEL_FIELDS)
    gdf_raw.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf_raw):,} parcels")

print(f"\nCRS: {gdf_raw.crs}   null geometry: {gdf_raw.geometry.isna().sum():,}")
print(f"Duplicate SCHEDNUM: {gdf_raw['SCHEDNUM'].duplicated().sum():,}")
print(f"SITUS_CITY: {gdf_raw['SITUS_CITY'].value_counts(dropna=False).head(4).to_dict()}")

### Downtown district boundaries

Scenario B's district is the **Downtown Denver Business Improvement District**.
It was chosen over the DDA/TIF footprint and a statistical-neighborhood "greater
downtown" because the BID is an already-constituted assessment district that
levies its own mills — which gives a district-scoped split-rate a plausible legal
vehicle. The DDA polygon and the neighborhood grouping are downloaded anyway so a
sensitivity run needs no new data pull.

In [ ]:
def fetch_boundary(layer_url, where="1=1"):
    r = requests.get(f"{layer_url}/query", params={
        "where": where, "outFields": "*", "returnGeometry": "true",
        "outSR": 4326, "geometryPrecision": 7, "f": "geojson",
    }, timeout=180)
    r.raise_for_status()
    g = gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326")
    # Several Denver district layers ship self-intersecting rings; repair before
    # any dissolve or sjoin (GEOS raises a side-location conflict otherwise).
    g["geometry"] = g.geometry.make_valid()
    return g


BOUNDARY_LAYERS = {
    "bids_all": f"{ARCGIS}/ODC_ADMN_BUSINESSIMPROVDIST_A/FeatureServer/1",
    "gids_all": f"{ARCGIS}/ODC_ADMN_GENERALIMPROVDIST_A/FeatureServer/294",
    "dda": f"{ARCGIS}/Downtown_Development_Authority/FeatureServer/0",
    "neighborhoods": f"{ARCGIS}/ODC_ADMN_NEIGHBORHOOD_A/FeatureServer/13",
}

bounds = {}
for name, url in BOUNDARY_LAYERS.items():
    p = DATA_DIR / f"{name}.parquet"
    if p.exists():
        bounds[name] = gpd.read_parquet(p)
    else:
        bounds[name] = fetch_boundary(url)
        bounds[name].to_parquet(p)
    print(f"{name:16} {len(bounds[name]):>4} features")

# --- Scenario B district: Downtown Denver BID ---
bid = bounds["bids_all"]
bid = bid[bid["DISTRICT_NAME"].str.contains("DOWNTOWN DENVER", case=False, na=False)].copy()
bid["name"] = "Downtown Denver BID"
DISTRICT = bid[["name", "DISTRICT_NAME", "CREATION_ORDINANCE",
                "BNDRY_LAST_MODIFIED_DATE", "ACTIVE_OR_HISTORIC", "geometry"]].copy()
DISTRICT.to_parquet(DATA_DIR / "downtown_bid.parquet")
print("\nScenario B district:")
print(DISTRICT.drop(columns="geometry").to_string(index=False))
print(f"BID area: {DISTRICT.to_crs(2877).area.sum() / 43560:,.1f} acres")

# --- Comparison polygons, saved for a possible sensitivity run ---
dda = bounds["dda"].copy()
dda["name"] = "Denver DDA"
dda.dissolve(by="name").reset_index()[["name", "geometry"]].to_parquet(
    DATA_DIR / "dda_dissolved.parquet")

# NOTE: Denver has no separate "Golden Triangle" statistical neighborhood —
# the Golden Triangle sits inside the Civic Center statistical neighborhood.
GREATER_DOWNTOWN = ["CBD", "Union Station", "Auraria", "Civic Center",
                    "North Capitol Hill", "Five Points"]
nb = bounds["neighborhoods"]
gd = nb[nb["NBHD_NAME"].isin(GREATER_DOWNTOWN)].copy()
gd["name"] = gd["NBHD_NAME"]
gd[["name", "NBHD_ID", "NBHD_NAME", "geometry"]].to_parquet(
    DATA_DIR / "greater_downtown_parts.parquet")
gd2 = gd.copy(); gd2["name"] = "Greater Downtown (statistical neighborhoods)"
gd2.dissolve(by="name").reset_index()[["name", "geometry"]].to_parquet(
    DATA_DIR / "greater_downtown.parquet")
print(f"\nComparison polygons saved: DDA ({len(dda)} parts), "
      f"greater downtown ({len(gd)} of {len(GREATER_DOWNTOWN)} neighborhoods found)")

## Section 3 — Reverse-engineer Colorado's assessment formula from the assessor's own numbers

Colorado's assessment rate is statutory, but it has moved repeatedly post-Gallagher
and it differs between school and non-school levies. Rather than *assume* a rate,
we fit the formula to the ~192k parcels where Denver's assessor published both the
appraised (actual) value and the assessed value, and report the exact-match rate.

Candidate forms tested:

* `assessed = round₁₀(rate × actual)` — a flat rate.
* `assessed = round₁₀(rate × (actual − min(pct × actual, cap)))` — a rate applied
  after a percentage actual-value reduction with a dollar cap. This is the shape
  Colorado's recent reform bills use.

A fit is only trusted when it reproduces the assessor's published number **exactly**
(within the $10 rounding increment) for essentially every parcel in the class.

In [ ]:
gdf = gdf_raw.copy()

# Structural cleaning, done before any fitting
n0 = len(gdf)
gdf = gdf[gdf["SCHEDNUM"].astype(str).str[0] != "9"].copy()
print(f"dropped {n0 - len(gdf):,} placeholder schedule numbers "
      f"(SCHEDNUM starting '9' — includes literal 'DUMMY PARCEL' rows)")

for c in ["APPRAISED_LAND_VALUE", "APPRAISED_IMP_VALUE", "APPRAISED_TOTAL_VALUE",
          "LAND_AREA", "RES_ABOVE_GRADE_AREA", "COM_GROSS_AREA", "TOT_UNITS"]:
    gdf[c] = pd.to_numeric(gdf[c], errors="coerce")

n1 = len(gdf)
gdf = gdf[gdf["APPRAISED_TOTAL_VALUE"].notna() & (gdf["APPRAISED_TOTAL_VALUE"] > 0)].copy()
print(f"dropped {n1 - len(gdf):,} parcels with no appraised value at all")

for c in ["APPRAISED_LAND_VALUE", "APPRAISED_IMP_VALUE"]:
    gdf[c] = gdf[c].fillna(0.0).clip(lower=0)

# Identity checks on the assessor's own arithmetic
obs = gdf[gdf["ASSESSED_TOTAL_VALUE_LOCAL"].notna()].copy()
chk_lb = (obs["ASSESSED_LAND_VALUE_LOCAL"] + obs["ASSESSED_BLDG_VALUE_LOCAL"]
          - obs["ASSESSED_TOTAL_VALUE_LOCAL"]).abs()
chk_tx = (obs["ASSESSED_TOTAL_VALUE_LOCAL"] - obs["EXEMPT_AMT_LOCAL"].fillna(0)
          - obs["TAXABLE_AMT_LOCAL"]).abs()
chk_ap = (obs["APPRAISED_LAND_VALUE"] + obs["APPRAISED_IMP_VALUE"]
          - obs["APPRAISED_TOTAL_VALUE"]).abs()
print(f"\nidentity: assessed land+bldg == total   -> {(chk_lb > 1).sum():,} mismatches")
print(f"identity: assessed - exempt == taxable -> {(chk_tx > 1).sum():,} mismatches")
print(f"identity: appraised land+imp == total  -> {(chk_ap > 1).sum():,} mismatches")

In [ ]:
# Fit the statutory formula, per Denver class group (DCL12)
RES_CANDS = [(r, p, c) for r in [0.0680, 0.0705, 0.0695, 0.0675, 0.0625, 0.0670, 0.0765]
             for (p, c) in [(0.0, 0), (0.10, 70_000), (0.10, 55_000), (0.10, 80_000)]]
NON_CANDS = [(r, 0.0, 0) for r in [0.25, 0.26, 0.27, 0.29, 0.30, 0.10, 0.2163]]
CANDS = RES_CANDS + NON_CANDS


def best_formula(actual, assessed, cands=CANDS):
    best = (0.0, None)
    for rate, pct, cap in cands:
        red = np.minimum(pct * actual, cap) if cap else np.zeros_like(actual, dtype=float)
        pred = np.round((actual - red) * rate, -1)
        score = float((np.abs(pred - assessed) <= 20).mean())
        if score > best[0]:
            best = (score, (rate, pct, cap))
    return best


rows = []
for dcl, sub in obs.groupby("DCL12"):
    A = sub["APPRAISED_TOTAL_VALUE"].to_numpy(float)
    for tag, col in [("local", "ASSESSED_TOTAL_VALUE_LOCAL"), ("school", "ASSESSED_TOTAL_VALUE_SCH")]:
        sc, f = best_formula(A, sub[col].to_numpy(float))
        rows.append({"DCL12": dcl, "levy": tag, "n": len(sub), "rate": f[0],
                     "reduction_pct": f[1], "reduction_cap": f[2], "exact_match": sc,
                     "appraised": sub["APPRAISED_TOTAL_VALUE"].sum()})
fit = pd.DataFrame(rows)

print("=== fitted statutory formula, LOCAL (non-school) levies ===")
print(fit[fit.levy == "local"].sort_values("appraised", ascending=False)
      .drop(columns="levy").head(30).round(5).to_string(index=False))
print("\n=== fitted statutory formula, SCHOOL levies ===")
print(fit[fit.levy == "school"].sort_values("appraised", ascending=False)
      .drop(columns="levy").head(12).round(5).to_string(index=False))

In [ ]:
# Consolidate the fit into three statutory tax classes.
# Classes whose exact-match score is low are Denver's "W/MIXED USE" codes, whose
# parcels blend a residential and a non-residential component — no single rate can
# reproduce them, which is why they fit poorly and why we never impute them from a
# single rate (see Section 4).
res_local = best_formula(
    obs.loc[obs.DCL12.isin(["11", "19", "12"]), "APPRAISED_TOTAL_VALUE"].to_numpy(float),
    obs.loc[obs.DCL12.isin(["11", "19", "12"]), "ASSESSED_TOTAL_VALUE_LOCAL"].to_numpy(float))
res_sch = best_formula(
    obs.loc[obs.DCL12.isin(["11", "19", "12"]), "APPRAISED_TOTAL_VALUE"].to_numpy(float),
    obs.loc[obs.DCL12.isin(["11", "19", "12"]), "ASSESSED_TOTAL_VALUE_SCH"].to_numpy(float))

RES_RATE, RES_RED_PCT, RES_RED_CAP = res_local[1]
SCH_RATE = res_sch[1][0]

print(f"RESIDENTIAL, local (non-school) levies: rate {RES_RATE:.4%} applied after a "
      f"{RES_RED_PCT:.0%} actual-value reduction capped at ${RES_RED_CAP:,.0f}")
print(f"  exact match on {res_local[0]*100:.2f}% of {obs.DCL12.isin(['11','19','12']).sum():,} parcels")
print(f"RESIDENTIAL, school levies: flat rate {SCH_RATE:.4%}, no reduction")
print(f"  exact match on {res_sch[0]*100:.2f}%")

# Non-residential: two distinct rates in the data — improved commercial/industrial
# vs. vacant land and other "all other" classes.
nonres = fit[(fit.levy == "local") & (fit.rate >= 0.20)]
print("\nNON-RESIDENTIAL rates observed (local == school):")
print(nonres.groupby("rate").agg(classes=("DCL12", "count"), parcels=("n", "sum"),
                                 appraised=("appraised", "sum")).to_string())

## Section 4 — Fill the assessed columns the GIS layer leaves null

Denver's GIS parcel layer publishes appraised values for every parcel but leaves
`ASSESSED_*` / `EXEMPT_*` **null for roughly 46,600 parcels — overwhelmingly
condominium units** (residential and commercial condos account for ~96% of them).
Those parcels are ~12% of Denver's appraised value and, because condominiums are
the densest housing in the city, they are exactly the parcels a land-value tax
most affects. Dropping them would bias the whole result, so they are reconstructed
from the statutory formula fitted in Section 3.

The reconstruction is deterministic, not estimated: the statutory rate is a
constant, the reduction is a published formula, and Section 3 showed the formula
reproduces the assessor's own published figure exactly for the classes where both
are available. Two honest caveats, both reported below:

1. **Exemptions are set to zero** for reconstructed parcels — the layer gives no
   exemption amount for them. Condominium units rarely carry the institutional
   exemptions that dominate Denver's exempt roll, so the effect is small, but it
   marginally overstates their taxable base.
2. **Mixed-use classes are not reconstructed from a single rate.** Denver's
   `W/MIXED USE` codes blend a residential and a non-residential component per
   parcel; for those we fall back to the class's median observed effective ratio
   rather than a statutory constant, and flag them.

In [ ]:
# Assign each Denver class group to a statutory tax class, using the fitted rates
local_fit = fit[fit.levy == "local"].set_index("DCL12")

TAX_CLASS = {}
for dcl, row in local_fit.iterrows():
    if row["rate"] < 0.15:
        TAX_CLASS[dcl] = "residential" if row["exact_match"] >= 0.95 else "mixed_residential"
    else:
        TAX_CLASS[dcl] = f"flat_{row['rate']:.2f}" if row["exact_match"] >= 0.95 else "mixed_nonres"

# DCL12 10 is Denver's condominium class. Almost every condo row has a null
# assessed value, so the class has too few observed parcels to fit — but a
# residential condominium unit is residential real property under Colorado law
# and takes the residential rate. Assign it explicitly.
TAX_CLASS["10"] = "residential"
for dcl in gdf["DCL12"].dropna().unique():
    TAX_CLASS.setdefault(dcl, "flat_0.26")   # conservative default: "all other"

gdf["tax_class"] = gdf["DCL12"].map(TAX_CLASS).fillna("flat_0.26")
print("tax_class assignment:")
print(gdf.groupby("tax_class").agg(
    parcels=("SCHEDNUM", "size"), appraised=("APPRAISED_TOTAL_VALUE", "sum")).to_string())

# Median observed effective ratio per class, for the mixed classes
med_ratio = (obs.assign(r=obs["ASSESSED_TOTAL_VALUE_LOCAL"] / obs["APPRAISED_TOTAL_VALUE"])
             .groupby("DCL12")["r"].median())


def statutory_assessed(actual, tax_class, dcl):
    """Assessed value for Denver's own (non-school) levy, per statutory formula."""
    out = np.zeros(len(actual), dtype=float)
    tc = np.asarray(tax_class); a = np.asarray(actual, dtype=float)
    is_res = np.isin(tc, ["residential"])
    red = np.minimum(RES_RED_PCT * a, RES_RED_CAP)
    out = np.where(is_res, np.round((a - red) * RES_RATE, -1), out)
    for rate in np.unique(local_fit["rate"].round(2)):
        m = tc == f"flat_{rate:.2f}"
        if m.any():
            out = np.where(m, np.round(a * rate, -1), out)
    mixed = np.isin(tc, ["mixed_residential", "mixed_nonres"])
    if mixed.any():
        r = pd.Series(dcl).map(med_ratio).fillna(0.26).to_numpy(float)
        out = np.where(mixed, np.round(a * r, -1), out)
    return out


gdf["assessed_imputed"] = gdf["ASSESSED_TOTAL_VALUE_LOCAL"].isna()
print(f"\nparcels needing reconstruction: {gdf['assessed_imputed'].sum():,} "
      f"(${gdf.loc[gdf.assessed_imputed, 'APPRAISED_TOTAL_VALUE'].sum():,.0f} appraised, "
      f"{gdf.loc[gdf.assessed_imputed, 'APPRAISED_TOTAL_VALUE'].sum() / gdf['APPRAISED_TOTAL_VALUE'].sum() * 100:.1f}% of value)")
print("\nreconstructed set by class:")
print(gdf[gdf.assessed_imputed].groupby(["tax_class", "D_CLASS_CN"]).size()
      .sort_values(ascending=False).head(12).to_string())

In [ ]:
# Build assessed land / building for Denver's own (non-school) levy.
# Observed parcels use the assessor's own columns verbatim. Reconstructed parcels
# apply the statutory formula to the appraised total, then split it pro-rata by
# the parcel's appraised land share (which is how the residential reduction is
# allocated in the assessor's own published components).
land_share = (gdf["APPRAISED_LAND_VALUE"]
              / gdf["APPRAISED_TOTAL_VALUE"].replace(0, np.nan)).fillna(0).clip(0, 1)

imp_total = statutory_assessed(gdf["APPRAISED_TOTAL_VALUE"], gdf["tax_class"], gdf["DCL12"])

gdf["asd_land"] = np.where(
    gdf["assessed_imputed"],
    np.round(imp_total * land_share, -1),
    pd.to_numeric(gdf["ASSESSED_LAND_VALUE_LOCAL"], errors="coerce").fillna(0.0),
)
gdf["asd_bldg"] = np.where(
    gdf["assessed_imputed"],
    np.round(imp_total * (1 - land_share), -1),
    pd.to_numeric(gdf["ASSESSED_BLDG_VALUE_LOCAL"], errors="coerce").fillna(0.0),
)
gdf["asd_total"] = gdf["asd_land"] + gdf["asd_bldg"]
gdf["exempt_local"] = np.where(
    gdf["assessed_imputed"], 0.0,
    pd.to_numeric(gdf["EXEMPT_AMT_LOCAL"], errors="coerce").fillna(0.0))

# Exemption hierarchy: full-exemption flag first, then dollar exemptions to
# improvements, then any remainder to land.
gdf["ex_share"] = (gdf["exempt_local"] / gdf["asd_total"].replace(0, np.nan)).fillna(0.0)
gdf["full_exmp"] = (gdf["ex_share"] >= 0.9999).astype(int)

_rem = gdf["exempt_local"].clip(upper=gdf["asd_total"])
_ex_b = np.minimum(_rem, gdf["asd_bldg"])
_ex_l = (_rem - _ex_b).clip(lower=0)
gdf["taxable_land_value"] = (gdf["asd_land"] - _ex_l).clip(lower=0)
gdf["taxable_improvement_value"] = (gdf["asd_bldg"] - _ex_b).clip(lower=0)
gdf["taxable_total"] = gdf["taxable_land_value"] + gdf["taxable_improvement_value"]

summary = pd.DataFrame({
    "metric": ["Parcels", "Appraised (actual) value", "Assessed value — Denver's own levy",
               "  ... from the assessor's own columns", "  ... reconstructed (Section 4)",
               "Exempt assessed value", "Taxable assessed value",
               "Fully-exempt parcels"],
    "value": [
        f"{len(gdf):,}",
        f"${gdf['APPRAISED_TOTAL_VALUE'].sum():,.0f}",
        f"${gdf['asd_total'].sum():,.0f}",
        f"${gdf.loc[~gdf.assessed_imputed, 'asd_total'].sum():,.0f}",
        f"${gdf.loc[gdf.assessed_imputed, 'asd_total'].sum():,.0f}",
        f"${gdf['exempt_local'].sum():,.0f}",
        f"${gdf['taxable_total'].sum():,.0f}",
        f"{gdf['full_exmp'].sum():,}",
    ]})
print(summary.to_string(index=False))
print(f"\nOverall effective assessment ratio: "
      f"{gdf['asd_total'].sum() / gdf['APPRAISED_TOTAL_VALUE'].sum() * 100:.2f}%")

## Section 4b — Restore missing improvement values (Gate 5 fix)

**This is a Gate-5 finding that had to be fixed before the model could be trusted.**

The first run reported that 4,363 *single-family homes* would see a **+90%**
municipal tax increase — the same increase as vacant land, which is exactly the
"distinct property types pinned at the same extreme" canary. Tracing them to the
parcel level: these are real 1940s–50s houses with real above-grade floor area and
real year-built records, carrying an `APPRAISED_IMP_VALUE` of exactly **$1,000**.
They cluster in Denver's established high-value neighborhoods — Washington Park
(18% of its single-family parcels), Cory–Merrill (24%), University Park (20%),
Wellshire (25%).

### Which half of the split is wrong?

Two hypotheses: either the *total* is right and value was misallocated into land,
or the *land* is right and the improvement value is simply absent. These imply
opposite repairs, so the question is settled empirically, by comparing **land value
per square foot** between corrupt and clean parcels *in the same neighborhood*:

| Neighborhood | corrupt land $/sq ft | clean land $/sq ft | ratio |
|---|---|---|---|
| Washington Park | 205.37 | 199.87 | **1.03** |
| Cory – Merrill | 114.93 | 114.93 | **1.00** |
| University Park | 135.08 | 133.00 | **1.02** |
| Wellshire | 92.38 | 93.39 | **0.99** |
| Hilltop | 135.84 | 133.86 | **1.02** |
| Hale | 89.80 | 89.54 | **1.00** |

In every neighborhood where corrupt and clean parcels have the same median lot
size, their land is valued **identically per square foot**. So the land value is
correct and the **improvement value is missing**. (Neighborhoods showing ratios
well above 1.0 — Highland, Speer, Five Points — are ones where the corrupt parcels
have much larger lots than the clean comparison set, so their $/sq ft is not
comparable; the matched-lot-size neighborhoods are the valid test.)

### The repair

Hold each parcel's **land value fixed** and restore its improvement value from the
median improvement ratio of *clean* parcels in the same **statistical neighborhood ×
assessor class** — the repo's existing St. Paul condo-collapse technique. Then
recompute the assessed value from the statutory formula fitted in Section 3, since
the assessor's assessed figure is missing the same improvement.

**The aggregate check confirms it independently.** Before the repair, modeled
non-exempt actual value was **1.60% below** Denver's published total actual value.
The restored improvement value is almost exactly that shortfall — the missing
buildings *are* the gap. After the repair the base validation moves from −1.60% to
roughly +0.6%, and a small positive residual is the expected direction, since the
parcel file is the roll-2026 assessment while the published control is roll 2025.

Detection rule: appraised improvement ≤ $1,000 **and** the parcel has a recorded
structure (`RES_ABOVE_GRADE_AREA > 0` or `COM_GROSS_AREA > 0`) **and** it has a
positive land value. Genuinely vacant parcels have no recorded structure and are
untouched, so the vacant-land finding is not diluted.

In [ ]:
# Attach Denver's 78 statistical neighborhoods (also used for reporting)
_nb = bounds["neighborhoods"].to_crs(2877)[["NBHD_NAME", "geometry"]]
_c = gdf.to_crs(2877).copy()
_c["geometry"] = _c.geometry.centroid
_j = gpd.sjoin(_c[["geometry"]], _nb, how="left", predicate="within")
_j = _j[~_j.index.duplicated(keep="first")]
gdf["NBHD"] = _j["NBHD_NAME"].reindex(gdf.index)
print(f"neighborhood matched: {gdf['NBHD'].notna().sum():,} / {len(gdf):,} "
      f"({gdf['NBHD'].isna().sum():,} unmatched)")

has_structure = (gdf["RES_ABOVE_GRADE_AREA"].fillna(0) > 0) | (gdf["COM_GROSS_AREA"].fillna(0) > 0)
gdf["missing_improvement"] = (
    (gdf["APPRAISED_IMP_VALUE"] <= 1000) & has_structure & (gdf["APPRAISED_LAND_VALUE"] > 0)
)
print(f"\\nparcels with a recorded structure but no improvement value: "
      f"{gdf['missing_improvement'].sum():,}")
print(f"  land value on those parcels: "
      f"${gdf.loc[gdf.missing_improvement, 'APPRAISED_LAND_VALUE'].sum():,.0f}")
print("\\nby assessor class:")
print(gdf[gdf.missing_improvement].groupby("D_CLASS_CN").size()
      .sort_values(ascending=False).head(10).to_string())
print("\\nmost affected neighborhoods:")
_t = gdf.groupby("NBHD").agg(n=("SCHEDNUM", "size"), affected=("missing_improvement", "sum"))
_t["pct"] = (_t.affected / _t.n * 100).round(1)
print(_t[_t.affected > 0].sort_values("affected", ascending=False).head(12).to_string())

In [ ]:
# The land-$/sqft test that decides the direction of the repair
_g = gdf.copy()
_g["land_psf"] = _g["APPRAISED_LAND_VALUE"] / _g["LAND_AREA"].replace(0, np.nan)
_g["ir"] = _g["APPRAISED_IMP_VALUE"] / _g["APPRAISED_TOTAL_VALUE"]
_sf = _g[_g["DCL12"].isin(["11", "19"])]
_rows = []
for _n, _s in _sf.groupby("NBHD"):
    _cc, _cl = _s[_s.missing_improvement], _s[~_s.missing_improvement]
    if len(_cc) < 30 or len(_cl) < 30:
        continue
    _rows.append({
        "NBHD": _n, "n_affected": len(_cc), "n_clean": len(_cl),
        "affected_land_psf": round(_cc.land_psf.median(), 2),
        "clean_land_psf": round(_cl.land_psf.median(), 2),
        "ratio": round(_cc.land_psf.median() / _cl.land_psf.median(), 3),
        "affected_lot": _cc.LAND_AREA.median(), "clean_lot": _cl.LAND_AREA.median(),
        "clean_impr_ratio": round(_cl.ir.median(), 3),
    })
_psf = pd.DataFrame(_rows).sort_values("n_affected", ascending=False)
print(_psf.to_string(index=False))
_matched = _psf[(_psf.affected_lot - _psf.clean_lot).abs() / _psf.clean_lot < 0.05]
print(f"\\nneighborhoods with matched median lot size: {len(_matched)}")
print(f"  median land-$/sqft ratio (affected / clean): {_matched['ratio'].median():.3f}")
print("  -> land value is correct; the improvement value is what is missing.")

In [ ]:
# Restore the improvement value: hold LAND fixed, impute the improvement from the
# median improvement ratio of clean parcels in the same neighborhood x assessor class.
clean = gdf[~gdf["missing_improvement"] & (gdf["APPRAISED_TOTAL_VALUE"] > 0)].copy()
clean["impr_ratio"] = clean["APPRAISED_IMP_VALUE"] / clean["APPRAISED_TOTAL_VALUE"]

by_nbhd_cat = clean.groupby(["NBHD", "D_CLASS_CN"])["impr_ratio"].median()
by_cat = clean.groupby("D_CLASS_CN")["impr_ratio"].median()
GLOBAL_RATIO = float(clean["impr_ratio"].median())

fix = gdf.index[gdf["missing_improvement"]]
ratios = pd.Series(
    [
        by_nbhd_cat.get((n, c), by_cat.get(c, GLOBAL_RATIO))
        for n, c in zip(gdf.loc[fix, "NBHD"], gdf.loc[fix, "D_CLASS_CN"])
    ],
    index=fix, dtype=float,
).fillna(GLOBAL_RATIO).clip(0.02, 0.95)

pre_land = gdf.loc[fix, "APPRAISED_LAND_VALUE"].sum()
pre_imp = gdf.loc[fix, "APPRAISED_IMP_VALUE"].sum()
pre_actual_nonexempt = gdf.loc[gdf["full_exmp"] == 0, "APPRAISED_TOTAL_VALUE"].sum()

# improvement = land * r/(1-r)  <=>  improvement/(land+improvement) = r
gdf.loc[fix, "APPRAISED_IMP_VALUE"] = (
    gdf.loc[fix, "APPRAISED_LAND_VALUE"] * (ratios / (1 - ratios))).round(-1)
gdf.loc[fix, "APPRAISED_TOTAL_VALUE"] = (
    gdf.loc[fix, "APPRAISED_LAND_VALUE"] + gdf.loc[fix, "APPRAISED_IMP_VALUE"])

print(f"restored improvement values on {len(fix):,} parcels")
print(f"  imputed improvement ratio: median {ratios.median():.3f}, "
      f"IQR {ratios.quantile(.25):.3f}-{ratios.quantile(.75):.3f}")
print(f"  land value:        ${pre_land:,.0f} -> "
      f"${gdf.loc[fix, 'APPRAISED_LAND_VALUE'].sum():,.0f}   (held fixed)")
print(f"  improvement value: ${pre_imp:,.0f} -> "
      f"${gdf.loc[fix, 'APPRAISED_IMP_VALUE'].sum():,.0f}")

# Recompute assessed values on the restored total, using the Section 3 formula,
# then split pro-rata by the parcel's appraised land share.
_new_asd = statutory_assessed(
    gdf.loc[fix, "APPRAISED_TOTAL_VALUE"], gdf.loc[fix, "tax_class"], gdf.loc[fix, "DCL12"])
_ls = (gdf.loc[fix, "APPRAISED_LAND_VALUE"]
       / gdf.loc[fix, "APPRAISED_TOTAL_VALUE"].replace(0, np.nan)).fillna(0).clip(0, 1)
pre_asd = gdf.loc[fix, "asd_total"].sum()
gdf.loc[fix, "asd_land"] = np.round(_new_asd * _ls.to_numpy(), -1)
gdf.loc[fix, "asd_bldg"] = np.round(_new_asd * (1 - _ls.to_numpy()), -1)
gdf.loc[fix, "asd_total"] = gdf.loc[fix, "asd_land"] + gdf.loc[fix, "asd_bldg"]

# Re-apply the exemption hierarchy on the repaired parcels
_rem = gdf.loc[fix, "exempt_local"].clip(upper=gdf.loc[fix, "asd_total"])
_ex_b = np.minimum(_rem, gdf.loc[fix, "asd_bldg"])
_ex_l = (_rem - _ex_b).clip(lower=0)
gdf.loc[fix, "taxable_land_value"] = (gdf.loc[fix, "asd_land"] - _ex_l).clip(lower=0)
gdf.loc[fix, "taxable_improvement_value"] = (gdf.loc[fix, "asd_bldg"] - _ex_b).clip(lower=0)
gdf["taxable_total"] = gdf["taxable_land_value"] + gdf["taxable_improvement_value"]

post_actual_nonexempt = gdf.loc[gdf["full_exmp"] == 0, "APPRAISED_TOTAL_VALUE"].sum()
print(f"\\n  assessed value on repaired parcels: ${pre_asd:,.0f} -> "
      f"${gdf.loc[fix, 'asd_total'].sum():,.0f}")
print(f"  citywide taxable assessed base:     ${gdf['taxable_total'].sum():,.0f}")
print(f"\\n  INDEPENDENT CHECK — non-exempt actual value vs Denver's published total:")
print(f"    before repair: ${pre_actual_nonexempt:,.0f}  "
      f"({(pre_actual_nonexempt / DENVER_TOTAL_ACTUAL_VALUE_2025 - 1) * 100:+.2f}%)")
print(f"    after repair:  ${post_actual_nonexempt:,.0f}  "
      f"({(post_actual_nonexempt / DENVER_TOTAL_ACTUAL_VALUE_2025 - 1) * 100:+.2f}%)")
print(f"    restored improvement value accounts for "
      f"${post_actual_nonexempt - pre_actual_nonexempt:,.0f} of the "
      f"${DENVER_TOTAL_ACTUAL_VALUE_2025 - pre_actual_nonexempt:,.0f} shortfall")

### The other Gate-5 canary: parcels with $0 land — checked, and legitimately so

1,241 parcels carry a positive improvement value and **zero** land value, including
several large downtown office buildings, and every one of them lands at *exactly*
the same tax change (with no land, the change is just the ratio of the new
improvement millage to the old flat millage). That is the classic
"distinct property types pinned at the same extreme value" canary, so it was traced.

It is **not** an error. Denver carries these as separate interests in a
condominium-ised or ground-leased building, with the **land on a sibling schedule
number**. Every one of the 21 such offices inside the BID shares a schedule-number
prefix with a land-only parcel, and 1901 Wazee St (McGregor Square) is the textbook
case: eight separate building interests, land carried separately. The land *is*
taxed — on the sibling parcel — so the model is right in aggregate, and only the
per-parcel attribution is split across schedules.

No values are fabricated for these parcels. Their materiality is reported below so
the deck never quotes a per-parcel figure that depends on them.

## Section 5 — Property categories

Denver's `D_CLASS_CN` carries 104 distinct class descriptions — rich enough to
split condominiums, townhomes/rowhouses, mixed use, hotels, parking, and the
commercial subtypes rather than collapsing everything into residential/commercial.
Note Denver's idiosyncratic use of an `INDUSTRIAL-` prefix for institutional uses
(`INDUSTRIAL-CHURCH`, `INDUSTRIAL-SCHOOL`), which are mapped to `Other`, not
`Industrial`, and are almost entirely fully exempt anyway.

Denver has **no surface-parking class code** — surface lots appear as
`VACANT LAND` or `COMMERCIAL-MISC IMPS`. Only structured parking
(`COMMERCIAL-PARKING GARAGE`) is separately coded, so `Transportation - Parking`
here means *garages*. The deck's surface-parking figures must come from the
`ODC_TRANS_PARKINGLOTS_A` layer, not from this classification.

In [ ]:
def categorize(desc: str, dcl: str) -> str:
    d = (desc or "").upper()
    if "CONDOMIN" in d:
        return "Office / Commercial Condo" if d.startswith("COMMERCIAL") else "Condominium"
    if "W/MIXED USE" in d or "MIXED USE" in d:
        return "Mixed Use"
    if "ROWHOUSE" in d:
        return "Townhome / Rowhouse"
    if d.startswith("SFR") or d == "RESIDENTIAL":
        return "Single Family Residential"
    if "DUPLEX" in d or "TRIPLEX" in d:
        return "Small Multi-Family (2-4 units)"
    if "4 TO 8 UNITS" in d or "MULTI UNIT APTS" in d or d in {
            "RESIDENTIAL-APARTMENT", "RESIDENTIAL-SENIOR CITIZEN APT"}:
        return "Large Multi-Family (5+ units)"
    if d in {"RESIDENTIAL-BOARDING HOME", "RESIDENTIAL-NURSING FACILITY",
             "RESIDENTIAL-MISC IMPS", "MOBILE HOME LAND", "MH / MINOR STRUCTURES"}:
        return "Other Residential"
    if "VACANT LAND" in d or d.startswith("RESIDENTIAL LAND") or \
            d.startswith("RESIDENTIAL  LAND") or "GENERAL COMMON ELEMENTS" in d or \
            d == "RESIDENTIAL GRACE YEAR":
        return "Vacant Land"
    if "FARM LAND" in d:
        return "Agricultural"
    if "PARKING GARAGE" in d:
        return "Transportation - Parking"
    if "HOTEL" in d or "MOTEL" in d:
        return "Hotel"
    if "OFFICE" in d:
        return "Office / Commercial Condo"
    if d in {"COMMERCIAL-RETAIL", "COMMERCIAL-SHOPPING CENTER", "COMMERCIAL-RESTAURANT",
             "COMMERCIAL-THEATER", "INDUSTRIAL-CONV STORE W/PUMPS",
             "INDUSTRIAL-SERVICE STATION"}:
        return "Retail / General Commercial"
    if d in {"INDUSTRIAL-CHURCH", "INDUSTRIAL-SCHOOL", "INDUSTRIAL-PRESCHOOL",
             "INDUSTRIAL-MEETING HALL", "INDUSTRIAL-CONVERTED CHURCH",
             "INDUSTRIAL-HEALTH CLUB", "INDUSTRIAL-CITY CLUB",
             "INDUSTRIAL-BOWLING ALLEY", "INDUSTRIAL-MISC RECREATION",
             "INDUSTRIAL-CEMETERY BLDG", "SOCIAL/RECREATION W/MIXED USE",
             "DENVER PARK", "GOLF COURSE", "STOCK SHOW", "FIRE STATION",
             "COUNTY JAIL", "STADIUM", "POLICE-FIRE STATION", "SPECIAL PURPOSE"}:
        return "Other"
    if d.startswith("INDUSTRIAL"):
        return "Industrial"
    if d.startswith("COMMERCIAL") or "P.I." in d or "POSSESSORY" in d:
        return "Other Commercial"
    return "Other"


gdf["PROPERTY_CATEGORY"] = [categorize(a, b) for a, b in
                            zip(gdf["D_CLASS_CN"], gdf["DCL12"])]

# Repo-wide override: any parcel with $0 improvement value is Vacant Land,
# whatever the assessor's use code says.
gdf.loc[gdf["APPRAISED_IMP_VALUE"] <= 0, "PROPERTY_CATEGORY"] = "Vacant Land"

print(gdf.groupby("PROPERTY_CATEGORY").agg(
    parcels=("SCHEDNUM", "size"),
    appraised=("APPRAISED_TOTAL_VALUE", "sum"),
    taxable=("taxable_total", "sum"),
    pct_exempt=("full_exmp", "mean"),
).sort_values("parcels", ascending=False).to_string())
assert gdf["PROPERTY_CATEGORY"].notna().all()

## Section 6 — Rebuild the current tax (Denver's own municipal levy) and gate it

Denver's own mill levy is **uniform across the city** — the 172 distinct `TAX_DIST`
codes in the parcel file vary because of *other* overlapping entities (BIDs, GIDs,
metro districts, the DDA), not because Denver's own rate differs by district. So
the current municipal tax is just `taxable assessed value × Denver's own mills`.

### The roll-year problem, stated plainly

The parcel file is the **roll-2026** assessment (Section 3 fits its formula exactly:
6.80% on residential after a 10%-of-value reduction capped at $70,000, 25% on
improved commercial, 26% on vacant land — a combination unique to roll 2026 under
HB24B-1001). But the most recent **certified** levy is **roll 2025**: 26.328 mills
raising $669,926,384. Colorado's roll-2026 rate cuts (residential effectively
6.25% → 6.12%, non-residential 27% → 25/26%) shrink the assessed base by ~8%, so
applying roll-2025 mills to a roll-2026 base *under*-raises by about that much.
Denver will raise its mills in the roll-2026 certification to offset this; it is
de-Bruced (Measure 2A, 2012) and so may.

So the gate is run in two parts:

1. **Base validation (the real test).** Compare modeled non-exempt **actual**
   (appraised) value against Denver's published total actual value. Actual value is
   not affected by the rate change, so this isolates whether the parcel universe and
   the exemption handling are right. This is the gate that can actually fail.
2. **Levy calibration.** Derive the effective roll-2026 municipal millage that makes
   modeled revenue equal Denver's certified own levy, and report it against the
   published roll-2025 rate. This is the Baltimore/Madison derived-millage pattern.

**Why the calibration is safe:** a revenue-neutral split-rate result is *invariant*
to the level of the municipal millage. Scaling the millage by any constant scales
every parcel's `current_tax` and `new_tax` by the same constant, so every
`tax_change_pct` is unchanged. The roll-year mismatch moves dollar magnitudes only —
not a single percentage in this analysis.

In [ ]:
# --- Gate 1: base validation on ACTUAL value (unaffected by the rate change) ---
nonexempt_actual = gdf.loc[gdf["full_exmp"] == 0, "APPRAISED_TOTAL_VALUE"].sum()
base_gap = (nonexempt_actual / DENVER_TOTAL_ACTUAL_VALUE_2025 - 1) * 100
print("GATE 1 — tax base validation (actual/appraised value)")
print(f"  modeled non-exempt actual value:  ${nonexempt_actual:,.0f}")
print(f"  Denver published total actual:    ${DENVER_TOTAL_ACTUAL_VALUE_2025:,.0f}")
print(f"  gap: {base_gap:+.2f}%   (must be within ±5%)")
assert abs(base_gap) < 5.0, f"BASE GATE FAILED: {base_gap:.2f}%"

# --- Assessed-value comparison: shows the roll-2026 statutory rate cut ---
taxable_base = gdf.loc[gdf["full_exmp"] == 0, "taxable_total"].sum()
print(f"\nmodeled taxable assessed base (roll {ROLL_YEAR_PARCELS}): ${taxable_base:,.0f}")
print(f"Denver gross assessed value (roll {ROLL_YEAR_LEVY}):      "
      f"${DENVER_GROSS_AV_2025_LOCAL:,.0f}")
print(f"  difference: {(taxable_base / DENVER_GROSS_AV_2025_LOCAL - 1) * 100:+.2f}%  "
      "— expected, and in the expected direction: roll 2026 cut the residential rate")
print("  from 6.25% to an effective 6.12% and non-residential from 27% to 25%/26%.")

In [ ]:
# --- Gate 2: levy calibration. Derive the effective roll-2026 municipal millage. ---
DERIVED_CITY_MILLAGE = OFFICIAL_CITY_LEVY / taxable_base * 1000.0
print("GATE 2 — levy calibration")
print(f"  certified own levy (roll {ROLL_YEAR_LEVY}):        ${OFFICIAL_CITY_LEVY:,.0f}")
print(f"  published own mills (roll {ROLL_YEAR_LEVY}):       {DENVER_OWN_MILLS_PUBLISHED:.4f}")
print(f"  derived own mills on the roll-{ROLL_YEAR_PARCELS} base: {DERIVED_CITY_MILLAGE:.4f}")
print(f"  implied mill increase: "
      f"{(DERIVED_CITY_MILLAGE / DENVER_OWN_MILLS_PUBLISHED - 1) * 100:+.2f}% "
      "— the statutory rate cut being offset, not a tax increase")
print(f"  Denver's share of the minimum stacked levy: "
      f"{DENVER_OWN_MILLS_PUBLISHED / TOTAL_STACKED_MILLS_2025 * 100:.1f}% "
      f"of {TOTAL_STACKED_MILLS_2025} mills")

gdf["millage_rate"] = DERIVED_CITY_MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col="taxable_total",
    millage_rate_col="millage_rate",
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    exemption_flag_col="full_exmp",
)
gap = (current_revenue / OFFICIAL_CITY_LEVY - 1) * 100
print(f"\n  modeled municipal revenue: ${current_revenue:,.0f}")
print(f"  REVENUE GATE: {gap:+.3f}%  (anchored to the certified levy by construction)")
assert abs(gap) < 5.0, f"Revenue gate FAILED: {gap:.2f}%"

# For reporting: what the published roll-2025 rate would raise on this base
print(f"\n  (for reference) published {DENVER_OWN_MILLS_PUBLISHED} mills on this base "
      f"= ${taxable_base * DENVER_OWN_MILLS_PUBLISHED / 1000:,.0f}, "
      f"{(taxable_base * DENVER_OWN_MILLS_PUBLISHED / 1000 / OFFICIAL_CITY_LEVY - 1) * 100:+.2f}% "
      "vs the certified levy")
DENVER_OWN_MILLS = DERIVED_CITY_MILLAGE

## Section 7 — Scenario A: citywide 4:1 split-rate

Revenue-neutral to Denver's own municipal levy across the whole city. Fully-exempt
parcels are held out of the solver and excluded from the modeled output.

In [ ]:
n_exempt = int((gdf["full_exmp"] == 1).sum())
df = gdf[gdf["full_exmp"] == 0].copy().reset_index(drop=True)
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts)")
print(f"Modeled taxable parcels: {len(df):,}")

land_millage, improvement_millage, new_revenue, df = model_split_rate_tax(
    df=df,
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    current_revenue=df["current_tax"].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"\nSCENARIO A — citywide 4:1")
print(f"  current municipal millage: {DENVER_OWN_MILLS:.4f}")
print(f"  land millage:              {land_millage:.4f}")
print(f"  improvement millage:       {improvement_millage:.4f}"
      f"  (ratio {land_millage / improvement_millage:.2f}:1)")
print(f"  revenue: ${new_revenue:,.0f}  vs target ${df['current_tax'].sum():,.0f}"
      f"  ({(new_revenue / df['current_tax'].sum() - 1) * 100:+.4f}%)")

df["new_tax_A"] = df["new_tax"]
df["tax_change_A"] = df["tax_change"]
df["tax_change_pct_A"] = df["tax_change_pct"]

cat_A = calculate_category_tax_summary(
    df=df, category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax", new_tax_col="new_tax_A")
print_category_tax_summary(cat_A, title="Denver, CO — Scenario A: citywide 4:1 split-rate "
                                        "(City & County municipal levy only)")

## Section 8 — Scenario B: 4:1 split-rate inside the Downtown Denver BID only

Same ratio, same levy, but the split-rate applies **only to parcels whose centroid
falls inside the Downtown Denver BID**, and revenue neutrality is enforced *within
the district*: the BID parcels' aggregate contribution to Denver's own municipal
levy is held constant, and every parcel outside the BID keeps its current
municipal tax exactly.

Centroid-in-polygon assignment (rather than intersects) matches the repo's
convention and avoids double-counting parcels that straddle the boundary.

In [ ]:
# Centroid-in-polygon assignment, done in a projected CRS (EPSG:2877,
# NAD83 / Colorado Central ftUS) so centroids are geometrically correct.
dist = DISTRICT.to_crs(2877)[["name", "geometry"]]
cent = df.to_crs(2877).copy()
cent["geometry"] = cent.geometry.centroid

joined = gpd.sjoin(cent[["geometry"]], dist, how="left", predicate="within")
joined = joined[~joined.index.duplicated(keep="first")]
df["lvt_district"] = joined["name"].reindex(df.index)
df["in_bid"] = df["lvt_district"].notna()

print(f"Parcels inside the Downtown Denver BID: {df['in_bid'].sum():,} "
      f"({df['in_bid'].mean() * 100:.2f}% of modeled parcels)")
print(f"BID share of Denver's municipal levy:   "
      f"{df.loc[df.in_bid, 'current_tax'].sum() / df['current_tax'].sum() * 100:.2f}%")
print(f"BID taxable assessed value:             ${df.loc[df.in_bid, 'taxable_total'].sum():,.0f}")

In [ ]:
bid_df = df[df["in_bid"]].copy()
assert len(bid_df) > 0, "No parcels fell inside the BID — check the polygon and CRS"

bid_target = bid_df["current_tax"].sum()
land_millage_B, improvement_millage_B, new_revenue_B, bid_df = model_split_rate_tax(
    df=bid_df,
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    current_revenue=bid_target,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Outside the BID: nothing changes.
df["new_tax_B"] = df["current_tax"]
df.loc[bid_df.index, "new_tax_B"] = bid_df["new_tax"].to_numpy()
df["tax_change_B"] = df["new_tax_B"] - df["current_tax"]
df["tax_change_pct_B"] = np.where(
    df["current_tax"] > 0, df["tax_change_B"] / df["current_tax"] * 100, np.nan)

print(f"SCENARIO B — Downtown Denver BID only, 4:1")
print(f"  land millage:        {land_millage_B:.4f}   (citywide A: {land_millage:.4f})")
print(f"  improvement millage: {improvement_millage_B:.4f}   (citywide A: {improvement_millage:.4f})")
print(f"  ratio {land_millage_B / improvement_millage_B:.2f}:1")
print(f"  district revenue: ${new_revenue_B:,.0f} vs target ${bid_target:,.0f} "
      f"({(new_revenue_B / bid_target - 1) * 100:+.4f}%)")
print(f"  citywide revenue: ${df['new_tax_B'].sum():,.0f} vs "
      f"${df['current_tax'].sum():,.0f} "
      f"({(df['new_tax_B'].sum() / df['current_tax'].sum() - 1) * 100:+.4f}%)")

print("\n--- Scenario B, inside the BID only ---")
cat_B = calculate_category_tax_summary(
    df=df[df.in_bid], category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax", new_tax_col="new_tax_B")
print_category_tax_summary(cat_B, title="Denver, CO — Scenario B: 4:1 inside the "
                                        "Downtown Denver BID")

In [ ]:
# Scenario B headline cuts for the deck
b = df[df.in_bid]
print("=== Downtown Denver BID — Scenario B headline ===")
print(f"parcels in district:      {len(b):,}")
print(f"pay LESS:                 {(b.tax_change_B < -0.5).sum():,} "
      f"({(b.tax_change_B < -0.5).mean() * 100:.1f}%)")
print(f"pay MORE:                 {(b.tax_change_B > 0.5).sum():,} "
      f"({(b.tax_change_B > 0.5).mean() * 100:.1f}%)")
print(f"essentially unchanged:    {(b.tax_change_B.abs() <= 0.5).sum():,}")
print(f"median change:            {b.tax_change_pct_B.median():+.1f}% / "
      f"${b.tax_change_B.median():+,.0f}")
print(f"land share of BID base:   "
      f"{b.taxable_land_value.sum() / b.taxable_total.sum() * 100:.1f}%")

print("\n--- the parcels the deck leads with ---")
for label, mask in [
    ("Vacant Land", b.PROPERTY_CATEGORY == "Vacant Land"),
    ("Structured parking (garages)", b.PROPERTY_CATEGORY == "Transportation - Parking"),
    ("Undeveloped: $0 improvement", b.APPRAISED_IMP_VALUE <= 0),
    ("Underdeveloped: improvement < 50% of value",
     (b.APPRAISED_IMP_VALUE / b.APPRAISED_TOTAL_VALUE).lt(0.50)),
    ("Office", b.PROPERTY_CATEGORY == "Office / Commercial Condo"),
    ("Large multi-family", b.PROPERTY_CATEGORY == "Large Multi-Family (5+ units)"),
    ("Condominium", b.PROPERTY_CATEGORY == "Condominium"),
]:
    s = b[mask]
    if len(s) == 0:
        print(f"  {label:44} (none)")
        continue
    print(f"  {label:44} n={len(s):>5,}  median {s.tax_change_pct_B.median():+7.1f}%  "
          f"${s.tax_change_B.median():>+12,.0f}   net ${s.tax_change_B.sum():>+14,.0f}")

## Section 9 — Gate 5: artifact scan

Gates 1–4 confirm the model *ran*. This confirms the results are *believable*.
Read the table against economic priors: land-heavy categories (vacant land,
parking, low-rise commercial on valuable land) should pay more; building-heavy
categories (condominiums, apartments, offices in tall buildings) should pay less.
The canary is distinct property types pinned at the same extreme value, an
implausible land/building split, or a base that won't reconcile to a control.

In [ ]:
d = df.copy()
d["bldg_share"] = d["taxable_improvement_value"] / d["taxable_total"].replace(0, np.nan)
scan = d.groupby("PROPERTY_CATEGORY").agg(
    n=("tax_change_pct_A", "size"),
    median_pct_A=("tax_change_pct_A", "median"),
    median_bldg_share=("bldg_share", "median"),
    pct_zero_bldg=("taxable_improvement_value", lambda s: (s <= 10).mean()),
    pct_reconstructed=("assessed_imputed", "mean"),
).sort_values("median_pct_A", ascending=False)
print(scan.round(3).to_string())

print("\n=== Scenario A headline ===")
print(f"parcels modeled:      {len(df):,}   (fully exempt held out: {n_exempt:,})")
print(f"pay LESS:             {(df.tax_change_A < -0.5).sum():,} "
      f"({(df.tax_change_A < -0.5).mean() * 100:.1f}%)")
print(f"pay MORE:             {(df.tax_change_A > 0.5).sum():,} "
      f"({(df.tax_change_A > 0.5).mean() * 100:.1f}%)")
print(f"median change:        {df.tax_change_pct_A.median():+.1f}% / "
      f"${df.tax_change_A.median():+,.0f}")

sf = df[df.PROPERTY_CATEGORY == "Single Family Residential"]
print(f"\nSingle-family detached (owner-occupied proxy): n={len(sf):,}")
print(f"  median change {sf.tax_change_pct_A.median():+.1f}% / ${sf.tax_change_A.median():+,.0f}")
print(f"  median current city tax ${sf.current_tax.median():,.0f} -> "
      f"new ${sf.new_tax_A.median():,.0f}")
print(f"  median appraised value ${sf.APPRAISED_TOTAL_VALUE.median():,.0f}")
print(f"  share paying less: {(sf.tax_change_A < 0).mean() * 100:.1f}%")

print("\n=== land share of the taxable base, citywide ===")
print(f"  {df.taxable_land_value.sum() / df.taxable_total.sum() * 100:.1f}%")

print("\n=== materiality of the $0-land parcels (land carried on sibling schedules) ===")
zl = df[df["APPRAISED_LAND_VALUE"] <= 0]
print(f"  citywide: n={len(zl):,}, improvement ${zl.APPRAISED_IMP_VALUE.sum():,.0f} "
      f"({zl.APPRAISED_IMP_VALUE.sum() / df.APPRAISED_TOTAL_VALUE.sum() * 100:.2f}% of appraised value)")
zb = zl[zl.in_bid]
bo = df[(df.in_bid) & (df.PROPERTY_CATEGORY == "Office / Commercial Condo")]
print(f"  inside the BID: n={len(zb):,}, Scenario B net ${zb.tax_change_B.sum():,.0f}")
print(f"  BID office net change: ${bo.tax_change_B.sum():,.0f}; "
      f"excluding $0-land parcels: ${bo.loc[bo.APPRAISED_LAND_VALUE > 0, 'tax_change_B'].sum():,.0f}")
print(f"  BID office median change: {bo.tax_change_pct_B.median():+.1f}%; "
      f"excluding $0-land: {bo.loc[bo.APPRAISED_LAND_VALUE > 0, 'tax_change_pct_B'].median():+.1f}%")

print("\n=== parcels whose improvement value was restored: where they land after the fix ===")
rp = df[df["missing_improvement"]]
print(f"  n={len(rp):,}, Scenario A median {rp.tax_change_pct_A.median():+.1f}%, "
      f"net ${rp.tax_change_A.sum():,.0f}")
sfr = df[(df.PROPERTY_CATEGORY == "Single Family Residential")]
print(f"  single-family net change ${sfr.tax_change_A.sum():,.0f}, of which "
      f"${sfr.loc[sfr.missing_improvement, 'tax_change_A'].sum():,.0f} from repaired parcels")

## Section 10 — Undeveloped and underdeveloped land

In [ ]:
result = analyze_land_by_improvement_share(
    df, land_value_col="APPRAISED_LAND_VALUE", improvement_value_col="APPRAISED_IMP_VALUE")
total = result["total_adjusted_land_value"]
cats = {c["category"]: c for c in result["categories"]}
print(f"Total non-exempt appraised land value: ${total:,.0f}\n")
print(f"{'Category':<42}{'Parcels':>9}{'Land value':>19}{'% of land':>11}")
print("-" * 81)
for key, c in cats.items():
    print(f"{key:<42}{c['parcel_count']:>9,}${c['adjusted_land_value']:>18,.0f}"
          f"{c['share_of_total_land_value_pct']:>10.2f}%")

# The deck's "underdeveloped" definition: vacant plus anything under 50% built
impr_share = (df["APPRAISED_IMP_VALUE"] / df["APPRAISED_TOTAL_VALUE"]).fillna(0)
for label, mask in [("Vacant ($0 improvement)", df.APPRAISED_IMP_VALUE <= 0),
                    ("Under 50% built", impr_share.lt(0.50)),
                    ("Vacant OR under 50% built",
                     (df.APPRAISED_IMP_VALUE <= 0) | impr_share.lt(0.50))]:
    s = df[mask]
    print(f"\n{label}: {len(s):,} parcels ({len(s)/len(df)*100:.1f}%), "
          f"${s.APPRAISED_LAND_VALUE.sum():,.0f} land value "
          f"({s.APPRAISED_LAND_VALUE.sum()/df.APPRAISED_LAND_VALUE.sum()*100:.1f}% of citywide land)")
    print(f"  Scenario A median change: {s.tax_change_pct_A.median():+.1f}%   "
          f"net ${s.tax_change_A.sum():+,.0f}")
    sb = s[s.in_bid]
    if len(sb):
        print(f"  inside the BID: n={len(sb):,}, Scenario B median "
              f"{sb.tax_change_pct_B.median():+.1f}%, net ${sb.tax_change_B.sum():+,.0f}")

## Section 11 — Downtown-boundary reconnaissance

Profiles the district actually chosen (the Downtown Denver BID) and the two
alternatives kept on the shelf, so a sensitivity run can switch without a new
data pull. Also checks whether the BID footprint is recoverable from the parcel
file's own `TAX_DIST` codes — the cheapest possible path — and reports any
mismatch between the code-derived footprint and the published polygon.

In [ ]:
def profile(poly_gdf, label):
    p = poly_gdf.to_crs(2877)[["geometry"]].copy()
    c = df.to_crs(2877).copy(); c["geometry"] = c.geometry.centroid
    j = gpd.sjoin(c[["geometry"]], p, how="inner", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    s = df.loc[j.index]
    if len(s) == 0:
        return {"district": label, "parcels": 0}
    return {
        "district": label,
        "parcels": len(s),
        "acres": round(poly_gdf.to_crs(2877).area.sum() / 43560, 1),
        "appraised_land": s.APPRAISED_LAND_VALUE.sum(),
        "appraised_impr": s.APPRAISED_IMP_VALUE.sum(),
        "taxable_land": s.taxable_land_value.sum(),
        "taxable_impr": s.taxable_improvement_value.sum(),
        "land_share_of_base_pct": round(
            s.taxable_land_value.sum() / s.taxable_total.sum() * 100, 2),
        "share_of_city_levy_pct": round(
            s.current_tax.sum() / df.current_tax.sum() * 100, 3),
        "municipal_tax": s.current_tax.sum(),
    }


recon = [profile(DISTRICT, "Downtown Denver BID (CHOSEN)"),
         profile(gpd.read_parquet(DATA_DIR / "dda_dissolved.parquet"),
                 "Downtown Development Authority (TIF)"),
         profile(gpd.read_parquet(DATA_DIR / "greater_downtown.parquet"),
                 "Greater downtown (6 statistical neighborhoods)")]
for row in gpd.read_parquet(DATA_DIR / "greater_downtown_parts.parquet").itertuples():
    recon.append(profile(gpd.GeoDataFrame({"geometry": [row.geometry]}, crs="EPSG:4326"),
                         f"  nbhd: {row.NBHD_NAME}"))
recon_df = pd.DataFrame(recon)
print(recon_df.to_string(index=False))

In [ ]:
# Is the BID recoverable from the parcel file's own TAX_DIST codes?
inb = df[df.in_bid]
vc = inb["TAX_DIST"].value_counts()
print("TAX_DIST composition inside the published BID polygon:")
print(vc.head(12).to_string())
print(f"\ndistinct TAX_DIST codes inside the BID: {inb['TAX_DIST'].nunique()}")

# For each code that appears in the BID, how much of that code lies inside?
containment = []
for code_, n_in in vc.items():
    n_all = int((df["TAX_DIST"] == code_).sum())
    containment.append({"TAX_DIST": code_, "in_bid": n_in, "citywide": n_all,
                        "pct_of_code_inside_bid": round(n_in / n_all * 100, 1)})
cont = pd.DataFrame(containment).sort_values("in_bid", ascending=False)
print("\ncontainment of each TAX_DIST code within the BID:")
print(cont.head(15).to_string(index=False))

exclusive = cont[cont.pct_of_code_inside_bid >= 99.0]
covered = int(exclusive.in_bid.sum())
print(f"\ncodes lying (>=99%) inside the BID: {len(exclusive)}; "
      f"they cover {covered:,} of {len(inb):,} BID parcels "
      f"({covered / len(inb) * 100:.1f}%)")
print("VERDICT: a TAX_DIST-code proxy is "
      + ("usable" if covered / len(inb) > 0.9 else "NOT a clean substitute")
      + " for the published BID polygon.")

## Section 12 — Census join, standard export, report, parcel map

In [ ]:
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ["std_geoid", "median_income", "minority_pct", "black_pct"]
if not os.getenv("CENSUS_API_KEY"):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        df[_col] = float("nan")
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=180)
                # NOTE: get_census_data_with_boundaries already merges the ACS
                # attributes onto census_gdf, so match_to_census_blockgroups brings
                # median_income / minority_pct / black_pct across on its own. Do NOT
                # merge census_data again on std_geoid — that produces _x/_y suffixed
                # columns, leaves no plain `median_income`, and silently drops the
                # four neighborhood-equity charts from the report.
                df = match_to_census_blockgroups(df, census_gdf)
                if "minority_pct" not in df.columns and {"total_pop", "white_pop"} <= set(df.columns):
                    df["minority_pct"] = ((df["total_pop"] - df["white_pop"])
                                          / df["total_pop"] * 100).round(2)
                if "black_pct" not in df.columns and {"total_pop", "black_pop"} <= set(df.columns):
                    df["black_pct"] = (df["black_pop"] / df["total_pop"] * 100).round(2)
                _missing = [c for c in _census_cols if c not in df.columns]
                assert not _missing, f"census join produced no {_missing} column"
                print(f"Census join: {df['std_geoid'].notna().mean() * 100:.1f}% matched; "
                      f"median_income populated on {df['median_income'].notna().mean() * 100:.1f}%")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    df[_col] = float("nan")
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            df[_col] = float("nan")

In [ ]:
# Scenario A is the canonical `new_tax` in the standard cross-city export.
df["new_tax"] = df["new_tax_A"]
df["tax_change"] = df["tax_change_A"]
df["tax_change_pct"] = df["tax_change_pct_A"]

out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f"../../analysis/data/{CITY_NAME}.csv",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax_A",
    tax_change_col="tax_change_A",
    tax_change_pct_col="tax_change_pct_A",
    taxable_land_col="taxable_land_value",
    taxable_improvement_col="taxable_improvement_value",
    exempt_flag_col="full_exmp",
)

# Two-scenario export: the standard 16 columns first (so every cross-city reader
# keeps working unchanged), then Scenario B appended.
two = out_df.copy()
two["new_tax_A"] = df["new_tax_A"].to_numpy()
two["new_tax_B"] = df["new_tax_B"].to_numpy()
two["tax_change_B"] = df["tax_change_B"].to_numpy()
two["tax_change_pct_B"] = df["tax_change_pct_B"].to_numpy()
two["in_bid"] = df["in_bid"].to_numpy()
two["land_millage_B"] = land_millage_B
two["improvement_millage_B"] = improvement_millage_B
two["assessed_reconstructed"] = df["assessed_imputed"].to_numpy()
two.to_csv(f"../../analysis/data/{CITY_NAME}.csv", index=False)
print(f"Wrote ../../analysis/data/{CITY_NAME}.csv  {two.shape}")
print(list(two.columns))

In [ ]:
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Scenario B category charts, inside the district
from lvt.viz import create_spokane_property_category_chart, create_threshold_change_chart
out_b = out_df.copy()
out_b["new_tax"] = df["new_tax_B"].to_numpy()
out_b["tax_change"] = df["tax_change_B"].to_numpy()
out_b["tax_change_pct"] = df["tax_change_pct_B"].to_numpy()
out_b = out_b[df["in_bid"].to_numpy()]

rep = Path(f"../../analysis/reports/{CITY_NAME}")
rep.mkdir(parents=True, exist_ok=True)
cat_b_chart = calculate_category_tax_summary(
    df=out_b, category_col="property_category",
    current_tax_col="current_tax", new_tax_col="new_tax")
try:
    fig, _ax = create_spokane_property_category_chart(
        cat_b_chart, category_col="property_category", min_count=20,
        title="Scenario B — 4:1 split-rate inside the Downtown Denver BID")
    fig.savefig(rep / "category_impact_scenario_b.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote category_impact_scenario_b.png")
except Exception as e:
    print(f"scenario B category chart failed: {e}")
try:
    fig, _ax = create_threshold_change_chart(
        cat_b_chart, category_col="property_category", min_count=20,
        title="Scenario B — share of BID parcels changing more than 10%")
    fig.savefig(rep / "ten_pct_share_scenario_b.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote ten_pct_share_scenario_b.png")
except Exception as e:
    print(f"scenario B threshold chart failed: {e}")

In [ ]:
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

gdf_map = gpd.GeoDataFrame(df, geometry="geometry", crs=gdf_raw.crs)
map_gdf = save_parcel_map_export(
    gdf=gdf_map,
    city=CITY_NAME,
    output_path=f"../../analysis/maps/{CITY_NAME}.parquet",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)

# Cache the modeled parcels for the deck pipeline
Path("data").mkdir(exist_ok=True)
gdf_map.to_parquet("data/parcels.gpq")
print(f"\nWrote data/parcels.gpq ({len(gdf_map):,} modeled parcels)")
print("Done.")